In [1]:
import numpy as np

In [35]:
import torch
from torch import nn 

In [65]:
import matplotlib.pyplot as plt

In [138]:
import random

In [39]:
class LayerNormalization(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 5e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=1, keepdim=True)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift

In [44]:
class SnakePlayerModel(nn.Module):

    def __init__(self, dim_in=16, dim_hidden=64, dim_out=4):
        super().__init__()
        input_layer = nn.Linear(dim_in, dim_hidden)
        hidden_layer = nn.Linear(dim_hidden, dim_hidden)
        output_layer = nn.Linear(dim_hidden, dim_out)
        self.network = nn.Sequential(
            input_layer,
            nn.ReLU(),
            hidden_layer,
            LayerNormalization(dim_hidden),
            nn.ReLU(),
            output_layer
        )

    def forward(self, x):
        return self.network(x)


In [45]:
snake_4x4_model = SnakePlayerModel(dim_in=16, dim_hidden=64, dim_out=4)

In [46]:
snake_4x4_model

SnakePlayerModel(
  (network): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): LayerNormalization()
    (4): ReLU()
    (5): Linear(in_features=64, out_features=4, bias=True)
  )
)

In [111]:
def plot_state(state):
    N = state.view(4,4)
    plt.figure(figsize=(4,4))
    plt.imshow(N, cmap='Blues')

    def to_text(v):
        if int(v) == 1:
            return 'T'
        elif int(v) == 2:
            return 'H'
        elif int(v) == 3:
            return 'F'
        else:
            return '';

    ax = plt.gca()
    ax.set_xticks(np.arange(-0.5, N.shape[1], 1), minor=False)
    ax.set_yticks(np.arange(-0.5, N.shape[0], 1), minor=False)
    ax.grid(True, color='black', linestyle='-', linewidth=0.5)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    for i in range(4):
        for j in range(4):
            plt.text(j, i, to_text(N[i,j].item()), ha="center", va="center", color="white")

In [10]:
import json

In [3]:
data_path = "/Users/deepakkumar/PycharmProjects/snake_player/history.data"

In [129]:
with open(data_path) as f:
    data = f.read()

In [130]:
lines = data.split("\n")

In [136]:
for line in lines[:1]:
    print(json.loads(line))
    state = torch.tensor(obj['state'])
    action = obj['action']
    reward = obj['reward']
    game_over = obj['done']
    action_predictions = snake_4x4_model(state)
    print(action_predictions)
    

{'state': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 3.0, 0.0]], 'action': 0, 'reward': -10, 'next_state': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 3.0, 0.0]], 'done': True}
tensor([[ 0.0382,  0.1417, -0.0129,  0.5682]], grad_fn=<AddmmBackward0>)


In [159]:
data = []

In [160]:
for i, line in enumerate(lines):
    try:
        obj = json.loads(line)
        data.append(obj)
    except json.JSONDecodeError:
        print("Error on line no: {} line=".format(i), line)

Error on line no: 1818 line= 


In [168]:
sample_data = random.sample(data, 16)

In [175]:
t1 = torch.tensor(data[0]['state'])
t1

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 3., 0.]])

In [176]:
t2 = torch.tensor(data[1]['state'])
t2

tensor([[0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 3., 0.]])